# Notebook 6 — RMS-Balanced FT (Colab A100)

**Purpose:** Mahapatra's direct gradient-balancing control (Section E of
his review). All parameters trainable, LR $=10^{-5}$ (matching Low-LR FT),
5 epochs -- but after every backward pass, the visual and language
gradient groups are rescaled to matched RMS magnitude before the
optimizer step. This directly forces balance every step, rather than
relying on architectural freezing (Staged FT) or implicit smoothing from
a low learning rate (Low-LR FT).

**IMPORTANT -- $R_t$ convention flag:** the existing `GradientTracker`
class (and this notebook, for consistency) computes
$R_t = \|g_{\text{language}}\| / \|g_{\text{visual}}\|$. This is the
OPPOSITE ratio to the paper's current Eq. (2), which defines
$R_t = \|g_\psi\|/\|g_\phi\|$ with $\psi=$visual. This does not affect
any cross-method comparison (every experiment uses the same convention),
but the paper's equation or its interpretive sentence needs a fix before
submission. Flagged here, tracked separately -- do not let it block
this notebook.

**Environment pins:** transformers==4.41.2, peft==0.11.1 (same as all
other new experiments).

**Structure:** shared setup (Cells 1-6), then three independent
dataset sections (UICD, RSICD, ROCOv2) you can run in sequence in one
session, or copy this notebook into separate tabs to run datasets in
parallel like you did for Joint-Schedule FT.

**Output:** `rms_balanced_ft_{dataset}_seed{42,0,123}.json` in
`/content/drive/MyDrive/DAMF/logs/`, with `rt_log` (pre-rescale, for
comparability with all other methods) and `rt_log_balanced` (post-
rescale, should sit near the balance point every step -- a sanity check
that the intervention worked as intended).

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Install pinned dependencies

In [ ]:
import subprocess, sys

def pip_install(pkg):
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg,
                    '-q', '--no-warn-script-location'], check=True)

pip_install('transformers==4.41.2')
pip_install('peft==0.11.1')
pip_install('pycocoevalcap')
pip_install('nltk')
pip_install('datasets')
pip_install('evaluate')
pip_install('kaggle')

print('Installations complete. RESTART RUNTIME NOW, then re-run cells 1-2.')

## IMPORTANT -- restart runtime once, then verify


In [ ]:
import transformers, peft
assert transformers.__version__ == '4.41.2', f'STOP: {transformers.__version__}. Restart runtime.'
assert peft.__version__ == '0.11.1', f'STOP: {peft.__version__}. Restart runtime.'
print('Versions verified: transformers 4.41.2, peft 0.11.1')

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('wordnet', quiet=True)
print('NLTK data downloaded.')

## 3. Imports, config, environment report

In [ ]:
import sys, os, gc, json, math, random, shutil as _shutil
from pathlib import Path
import numpy as np
import torch

from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from PIL import Image
from collections import defaultdict
from datasets import load_dataset

from transformers import BlipProcessor, BlipForConditionalGeneration
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

print('=' * 55)
print('ENVIRONMENT REPORT')
print('=' * 55)
print(f'Python       : {sys.version.split()[0]}')
print(f'PyTorch      : {torch.__version__}')
print(f'Transformers : {transformers.__version__}')
print(f'PEFT         : {peft.__version__}')
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU          : {props.name}  ({props.total_memory/1e9:.1f} GB)')
print(f'Active device: {DEVICE}')

OUT = '/content/drive/MyDrive/DAMF/logs'
os.makedirs(OUT, exist_ok=True)
SEEDS = [42, 0, 123]

CFG = {
    'batch_size': 16, 'beam_size': 3, 'num_workers': 2,
    'total_epochs': 5, 'lr': 1e-5,  # matches Low-LR FT exactly
    'weight_decay': 0.01,
    'rt_log_every_n_steps': 10, 'rt_epsilon': 1e-8,
}

print(f'\nCFG loaded. Output: {OUT}')
print(f"LR = {CFG['lr']} (identical to Low-LR FT -- only the gradient")
print('rescaling differs between the two methods).')

## 4. RMS-Balanced training step (the new mechanism)

No hooks needed here -- gradients are read directly from `.grad` after
`scaler.unscale_(optimizer)`, which is the standard PyTorch AMP pattern
for manually modifying gradients before the optimizer step.

In [ ]:
def get_param_groups(model):
    """Split trainable params into visual / language groups by name,
    using the same substring convention as GradientTracker elsewhere
    in this project (vision_model / text_decoder)."""
    vis_params, lang_params = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if 'vision_model' in name:
            vis_params.append(p)
        elif 'text_decoder' in name:
            lang_params.append(p)
    return vis_params, lang_params

def rms_of_params(params, eps=1e-8):
    sq_sum, n = 0.0, 0
    for p in params:
        if p.grad is not None:
            sq_sum += p.grad.detach().pow(2).sum().item()
            n += p.grad.numel()
    return math.sqrt(sq_sum / n) if n > 0 else 0.0

def rescale_grads_(params, factor):
    for p in params:
        if p.grad is not None:
            p.grad.mul_(factor)

def train_one_epoch_rms_balanced(model, loader, optimizer, scaler,
                                  vis_params, lang_params,
                                  step_counter=None, eps=1e-8):
    """Standard training step, but after backward() and before the
    optimizer step, visual and language gradient groups are rescaled
    to a common RMS target (geometric mean of the two group RMS's).

    Logs both:
      - rt_log: PRE-rescale ratio (lang_rms/vis_rms), for direct
        comparability with every other method's rt_log field.
      - rt_log_balanced: POST-rescale ratio, should sit near 1.0
        every step -- confirms the intervention worked as intended.
    """
    model.train()
    total_loss, n_batches = 0.0, 0
    rt_log, rt_log_balanced = [], []
    if step_counter is None:
        step_counter = [0]

    for batch in loader:
        inputs = {
            'pixel_values': batch['pixel_values'].to(DEVICE),
            'input_ids': batch['input_ids'].to(DEVICE),
            'attention_mask': batch['attention_mask'].to(DEVICE),
        }
        inputs['labels'] = inputs['input_ids'].clone()
        optimizer.zero_grad()

        with torch.amp.autocast('cuda'):
            loss = model(**inputs).loss
        scaler.scale(loss).backward()

        # Unscale BEFORE reading/modifying .grad -- standard AMP pattern
        scaler.unscale_(optimizer)

        rms_vis = rms_of_params(vis_params, eps)
        rms_lang = rms_of_params(lang_params, eps)
        raw_rt = rms_lang / (rms_vis + eps)

        # Rescale both groups toward their geometric-mean RMS
        target = math.sqrt((rms_vis + eps) * (rms_lang + eps))
        vis_factor = target / (rms_vis + eps)
        lang_factor = target / (rms_lang + eps)
        rescale_grads_(vis_params, vis_factor)
        rescale_grads_(lang_params, lang_factor)

        # Diagnostic: confirm post-rescale ratio is near 1
        rms_vis_post = rms_of_params(vis_params, eps)
        rms_lang_post = rms_of_params(lang_params, eps)
        balanced_rt = rms_lang_post / (rms_vis_post + eps)

        scaler.step(optimizer)  # grads already unscaled; scaler detects this
        scaler.update()

        total_loss += loss.item()
        n_batches += 1
        step_counter[0] += 1
        if step_counter[0] % CFG['rt_log_every_n_steps'] == 0:
            rt_log.append((step_counter[0], raw_rt))
            rt_log_balanced.append((step_counter[0], balanced_rt))

    return total_loss / max(n_batches, 1), rt_log, rt_log_balanced

print('RMS-Balanced training step defined.')

## 5. Shared metric/eval utilities (verbatim from your existing pipeline)

In [ ]:
def compute_bleu4(predictions, references):
    smoother = SmoothingFunction().method4
    pred_tokens = [p.split() for p in predictions]
    ref_tokens = [[r.split() for r in refs] for refs in references]
    return corpus_bleu(ref_tokens, pred_tokens, smoothing_function=smoother)

def compute_cider(predictions, references):
    try:
        from pycocoevalcap.cider.cider import Cider
        from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer
        gts = {i: [{'caption': r} for r in refs] for i, refs in enumerate(references)}
        res = {i: [{'caption': p}] for i, p in enumerate(predictions)}
        tok = PTBTokenizer()
        sc, _ = Cider().compute_score(tok.tokenize(gts), tok.tokenize(res))
        return float(sc)
    except Exception as e:
        print(f'  CIDEr error: {e}')
        return None

def compute_meteor(predictions, references):
    try:
        import evaluate as hf_evaluate
        meteor = hf_evaluate.load('meteor')
        flat_refs = [refs[0] for refs in references]
        result = meteor.compute(predictions=predictions, references=flat_refs)
        return float(result['meteor'])
    except Exception as e:
        print(f'  METEOR error: {e}')
        return None

def get_val_loss(model, loader):
    model.eval()
    total, n = 0.0, 0
    with torch.no_grad():
        for batch in loader:
            inputs = {
                'pixel_values': batch['pixel_values'].to(DEVICE),
                'input_ids': batch['input_ids'].to(DEVICE),
                'attention_mask': batch['attention_mask'].to(DEVICE),
            }
            inputs['labels'] = inputs['input_ids'].clone()
            with torch.amp.autocast('cuda'):
                total += model(**inputs).loss.item()
            n += 1
    return total / max(n, 1)

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def worker_init_fn(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed); random.seed(worker_seed)

def save_logs(logs, filename):
    path = os.path.join(OUT, filename)
    with open(path, 'w') as f:
        json.dump(logs, f, indent=2)
    print(f'  Saved: {filename}')
    return path

def check_disk_space(min_gb=2.0):
    free_gb = _shutil.disk_usage(OUT).free / 1e9
    if free_gb < min_gb:
        print(f'  !! DISK WARNING: only {free_gb:.2f} GB free.')
        return False
    return True

def save_checkpoint(model, filename):
    if not check_disk_space():
        return None
    path = os.path.join(OUT, filename)
    try:
        torch.save(model.state_dict(), path)
        return path
    except RuntimeError as e:
        print(f'  !! Checkpoint save failed ({filename}): {e}')
        return None

def delete_checkpoint(filename):
    path = os.path.join(OUT, filename)
    if os.path.exists(path):
        os.remove(path)

BLIP_PROCESSOR = BlipProcessor.from_pretrained('Salesforce/blip-image-captioning-base')
print('Shared utilities defined.')

## 6. Disk-space check before starting

Your Drive was down to ~1.1 GB free during the last run. Clean up stray checkpoints first if needed.

In [ ]:
total, used, free = _shutil.disk_usage('/content/drive/MyDrive')
print(f'Drive free space: {free/1e9:.2f} GB')

logs_dir = Path(OUT)
pt_files = sorted(logs_dir.glob('*.pt'), key=lambda p: -p.stat().st_size)
if pt_files:
    print(f'\n.pt checkpoint files still present ({len(pt_files)}):')
    total_mb = 0
    for p in pt_files:
        size_mb = p.stat().st_size / 1e6
        total_mb += size_mb
        print(f'  {p.name}: {size_mb:.0f} MB')
    print(f'\nTotal: {total_mb:.0f} MB. Consider deleting stale ones for')
    print('experiments that already show best_bleu4 in their .json (meaning')
    print('the run completed and the checkpoint should have auto-deleted).')
else:
    print('\nNo stray .pt checkpoints found. Clean.')

if free / 1e9 < 5.0:
    print('\n!! WARNING: less than 5 GB free. Strongly consider cleaning up')
    print('before launching ROCOv2 (largest dataset, longest training).')

---
# PART A -- UICD
---

## A1. Download UICD from Kaggle (skip if already downloaded this session)

In [ ]:
kaggle_json_src = Path('/content/drive/MyDrive/kaggle.json')
assert kaggle_json_src.exists(), 'kaggle.json not found in Drive.'
os.makedirs('/root/.kaggle', exist_ok=True)
_shutil.copy(kaggle_json_src, '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)

uicd_dir = Path('/content/uicd_data')
if not uicd_dir.exists() or not any(uicd_dir.iterdir()):
    uicd_dir.mkdir(exist_ok=True)
    subprocess.run(['kaggle', 'datasets', 'download',
                    '-d', 'kiranmuhammad/uicd-underwater-dataset',
                    '-p', str(uicd_dir), '--unzip'], check=True)

captions_files = list(uicd_dir.rglob('UIC-captions.txt'))
assert captions_files, 'UIC-captions.txt not found.'
UICD_CAPS = str(captions_files[0])
image_dirs = [d for d in captions_files[0].parent.iterdir()
             if d.is_dir() and 'image' in d.name.lower()]
UICD_IMAGES = str(image_dirs[0])
print(f'UICD ready. Images: {len(list(Path(UICD_IMAGES).glob("*")))}')

## A2. UICD dataset + loaders

In [ ]:
def load_uicd_captions(captions_path):
    image_captions = defaultdict(list)
    with open(captions_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split('#')
            if len(parts) < 2:
                continue
            img_name = parts[0].strip()
            cap_part = parts[1].strip()
            caption = cap_part.split(' ', 1)[1].strip() if ' ' in cap_part else cap_part
            if img_name and caption:
                image_captions[img_name].append(caption)
    return dict(image_captions)

uicd_image_captions = load_uicd_captions(UICD_CAPS)
_rng = random.Random(42)
_all = sorted(uicd_image_captions.keys())
_rng.shuffle(_all)
n = len(_all)
tr_end = int(0.70 * n)
va_end = int(0.85 * n)
UICD_SPLITS = {'train': _all[:tr_end], 'val': _all[tr_end:va_end], 'test': _all[va_end:]}
print(f"UICD split: train={len(UICD_SPLITS['train'])} val={len(UICD_SPLITS['val'])} "
      f"test={len(UICD_SPLITS['test'])}")

class UICDataset(Dataset):
    def __init__(self, image_list, image_captions, image_folder, processor,
                max_length=30, deterministic=False):
        self.image_list = image_list
        self.image_captions = image_captions
        self.image_folder = image_folder
        self.processor = processor
        self.max_length = max_length
        self.deterministic = deterministic

    def __len__(self):
        return len(self.image_list)

    def __getitem__(self, idx):
        img_name = self.image_list[idx]
        image = Image.open(os.path.join(self.image_folder, img_name)).convert('RGB')
        caption = (self.image_captions[img_name][0] if self.deterministic
                  else random.choice(self.image_captions[img_name]))
        inputs = self.processor(images=image, text=caption, return_tensors='pt',
                                padding='max_length', truncation=True,
                                max_length=self.max_length)
        return {
            'pixel_values': inputs['pixel_values'].squeeze(0),
            'input_ids': inputs['input_ids'].squeeze(0),
            'attention_mask': inputs['attention_mask'].squeeze(0),
            'image_name': img_name,
        }

g = torch.Generator(); g.manual_seed(0)
uicd_train_loader = DataLoader(
    UICDataset(UICD_SPLITS['train'], uicd_image_captions, UICD_IMAGES, BLIP_PROCESSOR,
              deterministic=False),
    batch_size=CFG['batch_size'], shuffle=True, num_workers=CFG['num_workers'],
    pin_memory=True, worker_init_fn=worker_init_fn, generator=g)
uicd_val_loader = DataLoader(
    UICDataset(UICD_SPLITS['val'], uicd_image_captions, UICD_IMAGES, BLIP_PROCESSOR,
              deterministic=True),
    batch_size=CFG['batch_size'], shuffle=False, num_workers=CFG['num_workers'],
    pin_memory=True)
print(f'UICD train batches: {len(uicd_train_loader)}  val: {len(uicd_val_loader)}')

@torch.no_grad()
def evaluate_blip_uicd(model, loader, processor, image_captions, cfg):
    model.eval()
    predictions, references = [], []
    for batch in loader:
        gen_ids = model.generate(pixel_values=batch['pixel_values'].to(DEVICE),
                                 max_length=30, num_beams=cfg['beam_size'])
        predictions.extend(processor.batch_decode(gen_ids, skip_special_tokens=True))
        for name in batch['image_name']:
            references.append(image_captions[name])
    bleu4 = compute_bleu4(predictions, references)
    cider = compute_cider(predictions, references)
    meteor = compute_meteor(predictions, references)
    return bleu4, cider, meteor, predictions, references

## A3. UICD RMS-Balanced FT runner

In [ ]:
def run_uicd_rms_balanced(seed):
    exp_name = f'rms_balanced_ft_uicd_seed{seed}'
    json_path = os.path.join(OUT, f'{exp_name}.json')

    if os.path.exists(json_path):
        with open(json_path) as f:
            logs = json.load(f)
        done = len(logs['bleu4_per_epoch'])
        if done >= CFG['total_epochs']:
            print(f'  [{exp_name}] already complete. Skipping.')
            return logs
    else:
        logs = {
            'experiment': exp_name, 'dataset': 'UICD', 'seed': seed,
            'method': 'rms_balanced_ft', 'lr': CFG['lr'], 'epochs': CFG['total_epochs'],
            'freeze_vision': False, 'freeze_language': False,
            'bleu4_per_epoch': [], 'cider_per_epoch': [], 'meteor_per_epoch': [],
            'train_loss_per_epoch': [], 'val_loss_per_epoch': [],
            'rt_log': [], 'rt_log_balanced': [],
        }
        done = 0

    seed_everything(seed)
    model = BlipForConditionalGeneration.from_pretrained(
        'Salesforce/blip-image-captioning-base').to(DEVICE)
    vis_params, lang_params = get_param_groups(model)
    print(f'  [{exp_name}] visual params: {len(vis_params)}  language params: {len(lang_params)}')

    ckpt_path = os.path.join(OUT, f'{exp_name}_last.pt')
    if done > 0 and os.path.exists(ckpt_path):
        model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
        print(f'  [{exp_name}] loaded checkpoint from epoch {done}')

    optimizer = AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
    scaler = torch.amp.GradScaler('cuda')
    step_counter = [done * len(uicd_train_loader)]
    best_bleu4 = max(logs['bleu4_per_epoch']) if logs['bleu4_per_epoch'] else 0.0

    for epoch in range(done + 1, CFG['total_epochs'] + 1):
        avg_loss, rt_log, rt_log_bal = train_one_epoch_rms_balanced(
            model, uicd_train_loader, optimizer, scaler,
            vis_params, lang_params, step_counter=step_counter)
        val_loss = get_val_loss(model, uicd_val_loader)
        bleu4, cider, meteor, preds, refs = evaluate_blip_uicd(
            model, uicd_val_loader, BLIP_PROCESSOR, uicd_image_captions, CFG)

        logs['train_loss_per_epoch'].append(round(avg_loss, 4))
        logs['val_loss_per_epoch'].append(round(val_loss, 4))
        logs['bleu4_per_epoch'].append(round(bleu4, 4))
        logs['cider_per_epoch'].append(round(cider, 4) if cider else None)
        logs['meteor_per_epoch'].append(round(meteor, 4) if meteor else None)
        logs['rt_log'].extend(rt_log)
        logs['rt_log_balanced'].extend(rt_log_bal)

        if bleu4 > best_bleu4:
            best_bleu4 = bleu4
            logs['best_bleu4'] = bleu4
            logs['best_cider'] = cider
            logs['best_meteor'] = meteor
            logs['best_epoch'] = epoch
            logs['best_predictions'] = preds[:20]
            logs['best_references'] = [r[:2] for r in refs[:20]]

        cs = f'{cider:.4f}' if cider else 'N/A'
        ms = f'{meteor:.4f}' if meteor else 'N/A'
        print(f"  [{exp_name}] epoch {epoch}/{CFG['total_epochs']} | "
              f'train={avg_loss:.4f} val={val_loss:.4f} BLEU-4={bleu4:.4f} '
              f'CIDEr={cs} METEOR={ms}')
        save_checkpoint(model, f'{exp_name}_last.pt')
        save_logs(logs, f'{exp_name}.json')

    del model
    gc.collect()
    torch.cuda.empty_cache()
    delete_checkpoint(f'{exp_name}_last.pt')
    print(f'  [{exp_name}] complete. Best BLEU-4: {best_bleu4:.4f}\n')
    return logs

print('run_uicd_rms_balanced defined.')

## A4. LAUNCH -- UICD, all 3 seeds

In [ ]:
print('=' * 60)
print('RMS-BALANCED FT -- UICD, all 3 seeds')
print('=' * 60)
uicd_results = {}
for seed in SEEDS:
    print(f"\n{'#'*60}\n# SEED {seed}\n{'#'*60}")
    uicd_results[seed] = run_uicd_rms_balanced(seed)

print('\nUICD ALL SEEDS COMPLETE')
for seed, logs in uicd_results.items():
    rt = logs.get('rt_log', [])
    rtb = logs.get('rt_log_balanced', [])
    print(f"seed {seed}: best_bleu4={logs.get('best_bleu4'):.4f}  "
          f'raw_rt entries={len(rt)}  balanced_rt entries={len(rtb)}')
    if rtb:
        vals = [v for _, v in rtb]
        print(f'   balanced_rt range: [{min(vals):.3f}, {max(vals):.3f}] '
              f'(should hover near 1.0 -- confirms rescaling worked)')

---
# PART B -- RSICD
---

## B1. RSICD dataset + loaders

In [ ]:
print('Loading RSICD from HuggingFace...')
rsicd_raw = load_dataset('arampacha/rsicd')
print(f"RSICD: {len(rsicd_raw['train'])} train / {len(rsicd_raw['valid'])} val")

class RSICDDataset(Dataset):
    def __init__(self, hf_split, processor, max_length=30, deterministic=False):
        self.data = hf_split
        self.processor = processor
        self.max_length = max_length
        self.deterministic = deterministic

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        image = item['image'].convert('RGB')
        caption = (item['captions'][0] if self.deterministic
                  else random.choice(item['captions']))
        inputs = self.processor(images=image, text=caption, return_tensors='pt',
                                padding='max_length', truncation=True,
                                max_length=self.max_length)
        return {
            'pixel_values': inputs['pixel_values'].squeeze(0),
            'input_ids': inputs['input_ids'].squeeze(0),
            'attention_mask': inputs['attention_mask'].squeeze(0),
            'captions': item['captions'], 'filename': item['filename'],
        }

def rsicd_collate(batch):
    return {
        'pixel_values': torch.stack([b['pixel_values'] for b in batch]),
        'input_ids': torch.stack([b['input_ids'] for b in batch]),
        'attention_mask': torch.stack([b['attention_mask'] for b in batch]),
        'captions': [b['captions'] for b in batch],
        'filename': [b['filename'] for b in batch],
    }

rsicd_train_loader = DataLoader(
    RSICDDataset(rsicd_raw['train'], BLIP_PROCESSOR, deterministic=False),
    batch_size=CFG['batch_size'], shuffle=True, num_workers=CFG['num_workers'],
    pin_memory=True, collate_fn=rsicd_collate, worker_init_fn=worker_init_fn)
rsicd_val_loader = DataLoader(
    RSICDDataset(rsicd_raw['valid'], BLIP_PROCESSOR, deterministic=True),
    batch_size=CFG['batch_size'], shuffle=False, num_workers=CFG['num_workers'],
    pin_memory=True, collate_fn=rsicd_collate)
print(f'RSICD train batches: {len(rsicd_train_loader)}  val: {len(rsicd_val_loader)}')

@torch.no_grad()
def evaluate_blip_rsicd(model, loader, processor, cfg):
    model.eval()
    predictions, references = [], []
    for batch in loader:
        gen_ids = model.generate(pixel_values=batch['pixel_values'].to(DEVICE),
                                 max_length=30, num_beams=cfg['beam_size'])
        predictions.extend(processor.batch_decode(gen_ids, skip_special_tokens=True))
        references.extend(batch['captions'])
    bleu4 = compute_bleu4(predictions, references)
    cider = compute_cider(predictions, references)
    meteor = compute_meteor(predictions, references)
    return bleu4, cider, meteor, predictions, references

## B2. RSICD RMS-Balanced FT runner

In [ ]:
def run_rsicd_rms_balanced(seed):
    exp_name = f'rms_balanced_ft_rsicd_seed{seed}'
    json_path = os.path.join(OUT, f'{exp_name}.json')

    if os.path.exists(json_path):
        with open(json_path) as f:
            logs = json.load(f)
        done = len(logs['bleu4_per_epoch'])
        if done >= CFG['total_epochs']:
            print(f'  [{exp_name}] already complete. Skipping.')
            return logs
    else:
        logs = {
            'experiment': exp_name, 'dataset': 'RSICD', 'seed': seed,
            'method': 'rms_balanced_ft', 'lr': CFG['lr'], 'epochs': CFG['total_epochs'],
            'freeze_vision': False, 'freeze_language': False,
            'bleu4_per_epoch': [], 'cider_per_epoch': [], 'meteor_per_epoch': [],
            'train_loss_per_epoch': [], 'val_loss_per_epoch': [],
            'rt_log': [], 'rt_log_balanced': [],
        }
        done = 0

    seed_everything(seed)
    model = BlipForConditionalGeneration.from_pretrained(
        'Salesforce/blip-image-captioning-base').to(DEVICE)
    vis_params, lang_params = get_param_groups(model)

    ckpt_path = os.path.join(OUT, f'{exp_name}_last.pt')
    if done > 0 and os.path.exists(ckpt_path):
        model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
        print(f'  [{exp_name}] loaded checkpoint from epoch {done}')

    optimizer = AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
    scaler = torch.amp.GradScaler('cuda')
    step_counter = [done * len(rsicd_train_loader)]
    best_bleu4 = max(logs['bleu4_per_epoch']) if logs['bleu4_per_epoch'] else 0.0

    for epoch in range(done + 1, CFG['total_epochs'] + 1):
        avg_loss, rt_log, rt_log_bal = train_one_epoch_rms_balanced(
            model, rsicd_train_loader, optimizer, scaler,
            vis_params, lang_params, step_counter=step_counter)
        val_loss = get_val_loss(model, rsicd_val_loader)
        bleu4, cider, meteor, preds, refs = evaluate_blip_rsicd(
            model, rsicd_val_loader, BLIP_PROCESSOR, CFG)

        logs['train_loss_per_epoch'].append(round(avg_loss, 4))
        logs['val_loss_per_epoch'].append(round(val_loss, 4))
        logs['bleu4_per_epoch'].append(round(bleu4, 4))
        logs['cider_per_epoch'].append(round(cider, 4) if cider else None)
        logs['meteor_per_epoch'].append(round(meteor, 4) if meteor else None)
        logs['rt_log'].extend(rt_log)
        logs['rt_log_balanced'].extend(rt_log_bal)

        if bleu4 > best_bleu4:
            best_bleu4 = bleu4
            logs['best_bleu4'] = bleu4
            logs['best_cider'] = cider
            logs['best_meteor'] = meteor
            logs['best_epoch'] = epoch
            logs['best_predictions'] = preds[:20]
            logs['best_references'] = [r[:2] for r in refs[:20]]

        cs = f'{cider:.4f}' if cider else 'N/A'
        ms = f'{meteor:.4f}' if meteor else 'N/A'
        print(f"  [{exp_name}] epoch {epoch}/{CFG['total_epochs']} | "
              f'train={avg_loss:.4f} val={val_loss:.4f} BLEU-4={bleu4:.4f} '
              f'CIDEr={cs} METEOR={ms}')
        save_checkpoint(model, f'{exp_name}_last.pt')
        save_logs(logs, f'{exp_name}.json')

    del model
    gc.collect()
    torch.cuda.empty_cache()
    delete_checkpoint(f'{exp_name}_last.pt')
    print(f'  [{exp_name}] complete. Best BLEU-4: {best_bleu4:.4f}\n')
    return logs

print('run_rsicd_rms_balanced defined.')

## B3. LAUNCH -- RSICD, all 3 seeds

In [ ]:
print('=' * 60)
print('RMS-BALANCED FT -- RSICD, all 3 seeds')
print('=' * 60)
rsicd_results = {}
for seed in SEEDS:
    print(f"\n{'#'*60}\n# SEED {seed}\n{'#'*60}")
    rsicd_results[seed] = run_rsicd_rms_balanced(seed)

print('\nRSICD ALL SEEDS COMPLETE')
for seed, logs in rsicd_results.items():
    rtb = logs.get('rt_log_balanced', [])
    print(f"seed {seed}: best_bleu4={logs.get('best_bleu4'):.4f}")
    if rtb:
        vals = [v for _, v in rtb]
        print(f'   balanced_rt range: [{min(vals):.3f}, {max(vals):.3f}]')

---
# PART C -- ROCOv2
---

## C1. ROCOv2 dataset + loaders

In [ ]:
print('Loading ROCOv2 (full dataset) from HuggingFace...')
rocov2_raw = load_dataset('eltorio/ROCOv2-radiology')
print(f"ROCOv2: {len(rocov2_raw['train'])} train / {len(rocov2_raw['validation'])} val")

ROCOV2_MAX_LENGTH = 40  # radiology captions run longer -- preserved from original notebook

class ROCOv2Dataset(Dataset):
    def __init__(self, hf_split, processor, max_length):
        self.data = hf_split
        self.processor = processor
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        image = item['image'].convert('RGB')
        caption = item['caption']
        inputs = self.processor(images=image, text=caption, return_tensors='pt',
                                padding='max_length', truncation=True,
                                max_length=self.max_length)
        return {
            'pixel_values': inputs['pixel_values'].squeeze(0),
            'input_ids': inputs['input_ids'].squeeze(0),
            'attention_mask': inputs['attention_mask'].squeeze(0),
            'caption': caption, 'image_id': item['image_id'],
        }

def rocov2_collate(batch):
    return {
        'pixel_values': torch.stack([b['pixel_values'] for b in batch]),
        'input_ids': torch.stack([b['input_ids'] for b in batch]),
        'attention_mask': torch.stack([b['attention_mask'] for b in batch]),
        'captions': [[b['caption']] for b in batch],
        'image_id': [b['image_id'] for b in batch],
    }

rocov2_train_loader = DataLoader(
    ROCOv2Dataset(rocov2_raw['train'], BLIP_PROCESSOR, ROCOV2_MAX_LENGTH),
    batch_size=CFG['batch_size'], shuffle=True, num_workers=CFG['num_workers'],
    pin_memory=True, collate_fn=rocov2_collate, worker_init_fn=worker_init_fn)
rocov2_val_loader = DataLoader(
    ROCOv2Dataset(rocov2_raw['validation'], BLIP_PROCESSOR, ROCOV2_MAX_LENGTH),
    batch_size=CFG['batch_size'], shuffle=False, num_workers=CFG['num_workers'],
    pin_memory=True, collate_fn=rocov2_collate)
print(f'ROCOv2 train batches: {len(rocov2_train_loader)}  val: {len(rocov2_val_loader)}')

@torch.no_grad()
def evaluate_blip_rocov2(model, loader, processor, cfg):
    model.eval()
    predictions, references = [], []
    for batch in loader:
        gen_ids = model.generate(pixel_values=batch['pixel_values'].to(DEVICE),
                                 max_length=ROCOV2_MAX_LENGTH, num_beams=cfg['beam_size'])
        predictions.extend(processor.batch_decode(gen_ids, skip_special_tokens=True))
        references.extend(batch['captions'])
    bleu4 = compute_bleu4(predictions, references)
    cider = compute_cider(predictions, references)
    meteor = compute_meteor(predictions, references)
    return bleu4, cider, meteor, predictions, references

## C2. ROCOv2 RMS-Balanced FT runner

In [ ]:
def run_rocov2_rms_balanced(seed):
    exp_name = f'rms_balanced_ft_rocov2_seed{seed}'
    json_path = os.path.join(OUT, f'{exp_name}.json')

    if os.path.exists(json_path):
        with open(json_path) as f:
            logs = json.load(f)
        done = len(logs['bleu4_per_epoch'])
        if done >= CFG['total_epochs']:
            print(f'  [{exp_name}] already complete. Skipping.')
            return logs
    else:
        logs = {
            'experiment': exp_name, 'dataset': 'ROCOv2', 'seed': seed,
            'method': 'rms_balanced_ft', 'lr': CFG['lr'], 'epochs': CFG['total_epochs'],
            'freeze_vision': False, 'freeze_language': False,
            'bleu4_per_epoch': [], 'cider_per_epoch': [], 'meteor_per_epoch': [],
            'train_loss_per_epoch': [], 'val_loss_per_epoch': [],
            'rt_log': [], 'rt_log_balanced': [],
        }
        done = 0

    seed_everything(seed)
    model = BlipForConditionalGeneration.from_pretrained(
        'Salesforce/blip-image-captioning-base').to(DEVICE)
    vis_params, lang_params = get_param_groups(model)

    ckpt_path = os.path.join(OUT, f'{exp_name}_last.pt')
    if done > 0 and os.path.exists(ckpt_path):
        model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
        print(f'  [{exp_name}] loaded checkpoint from epoch {done}')

    optimizer = AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
    scaler = torch.amp.GradScaler('cuda')
    step_counter = [done * len(rocov2_train_loader)]
    best_bleu4 = max(logs['bleu4_per_epoch']) if logs['bleu4_per_epoch'] else 0.0

    for epoch in range(done + 1, CFG['total_epochs'] + 1):
        avg_loss, rt_log, rt_log_bal = train_one_epoch_rms_balanced(
            model, rocov2_train_loader, optimizer, scaler,
            vis_params, lang_params, step_counter=step_counter)
        val_loss = get_val_loss(model, rocov2_val_loader)
        bleu4, cider, meteor, preds, refs = evaluate_blip_rocov2(
            model, rocov2_val_loader, BLIP_PROCESSOR, CFG)

        logs['train_loss_per_epoch'].append(round(avg_loss, 4))
        logs['val_loss_per_epoch'].append(round(val_loss, 4))
        logs['bleu4_per_epoch'].append(round(bleu4, 4))
        logs['cider_per_epoch'].append(round(cider, 4) if cider else None)
        logs['meteor_per_epoch'].append(round(meteor, 4) if meteor else None)
        logs['rt_log'].extend(rt_log)
        logs['rt_log_balanced'].extend(rt_log_bal)

        if bleu4 > best_bleu4:
            best_bleu4 = bleu4
            logs['best_bleu4'] = bleu4
            logs['best_cider'] = cider
            logs['best_meteor'] = meteor
            logs['best_epoch'] = epoch
            logs['best_predictions'] = preds[:20]
            logs['best_references'] = refs[:20]

        cs = f'{cider:.4f}' if cider else 'N/A'
        ms = f'{meteor:.4f}' if meteor else 'N/A'
        print(f"  [{exp_name}] epoch {epoch}/{CFG['total_epochs']} | "
              f'train={avg_loss:.4f} val={val_loss:.4f} BLEU-4={bleu4:.4f} '
              f'CIDEr={cs} METEOR={ms}')
        save_checkpoint(model, f'{exp_name}_last.pt')
        save_logs(logs, f'{exp_name}.json')

    del model
    gc.collect()
    torch.cuda.empty_cache()
    delete_checkpoint(f'{exp_name}_last.pt')
    print(f'  [{exp_name}] complete. Best BLEU-4: {best_bleu4:.4f}\n')
    return logs

print('run_rocov2_rms_balanced defined.')

## C3. LAUNCH -- ROCOv2, all 3 seeds

Slowest of the three (largest dataset). Consider running this in its own
session/tab in parallel with Parts A/B if you copy this notebook, the way
you did for Joint-Schedule FT.

In [ ]:
print('=' * 60)
print('RMS-BALANCED FT -- ROCOv2, all 3 seeds')
print('=' * 60)
rocov2_results = {}
for seed in SEEDS:
    print(f"\n{'#'*60}\n# SEED {seed}\n{'#'*60}")
    rocov2_results[seed] = run_rocov2_rms_balanced(seed)

print('\nROCOv2 ALL SEEDS COMPLETE')
for seed, logs in rocov2_results.items():
    rtb = logs.get('rt_log_balanced', [])
    print(f"seed {seed}: best_bleu4={logs.get('best_bleu4'):.4f}")
    if rtb:
        vals = [v for _, v in rtb]
        print(f'   balanced_rt range: [{min(vals):.3f}, {max(vals):.3f}]')

print('\n' + '=' * 60)
print('ALL THREE DATASETS COMPLETE FOR RMS-BALANCED FT')
print('=' * 60)
print('Next: Multimodal LoRA notebook, then extend Notebook 1 analysis')
print('to include both new methods across all three datasets.')